**Setup & Installation**

Mounting Google Drive and installing dependencies (Transformers, etc).

In [2]:
!pip install -q transformers datasets scikit-learn


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


**1. Load Dataset**

Loading the generated FAQ JSON datasets: General, Business Dashboard, and KYC.

In [9]:
import json
import os
import pandas as pd

qna_bot_folder_path = './dataset/'
files_to_load = ['Q&A General.json', 'Q&A Business Dashboard.json', 'Q&A KYC.json']

all_data = []

for filename in files_to_load:
    file_path = os.path.join(qna_bot_folder_path, filename)
    if os.path.exists(file_path):
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            for item in data.get('data', []):
                # Append query, intent, answer
                all_data.append({
                    'query': item.get('question', ''),
                    'intent': item.get('intent', 'unknown'),
                    'answer': item.get('answer', '')
                })
        print(f"Loaded {filename}")
    else:
        print(f"File not found: {file_path}")

df = pd.DataFrame(all_data)
print(f"\nTotal dataset size: {len(df)}")
df.head()

Loaded Q&A General.json
Loaded Q&A Business Dashboard.json
Loaded Q&A KYC.json

Total dataset size: 4823


,query,intent,answer
0,Apa data akun saya aman?,account_data_security,"Ya, data anda tersimpan aman dan tidak dipergu..."
1,Bagaimana dengan data saya dijaga dengan aman?,account_data_security,"Ya, data anda tersimpan aman dan tidak dipergu..."
2,Apakah privasi saya aman? di aplikasi ini?,account_data_security,"Ya, data anda tersimpan aman dan tidak dipergu..."
3,Bagaimana dengan data pribadi dijaga dengan aman?,account_data_security,"Ya, data anda tersimpan aman dan tidak dipergu..."
4,Apakah data pribadi terlindungi? di sini?,account_data_security,"Ya, data anda tersimpan aman dan tidak dipergu..."


In [10]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# Ensure consistent mapping
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['intent'])

num_labels = len(label_encoder.classes_)
print(f"Total unique intents: {num_labels}")

# Train-Validation Split
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['query'].tolist(), 
    df['label'].tolist(), 
    test_size=0.15, 
    random_state=42
)

print(f"Train size: {len(train_texts)}, Validation size: {len(val_texts)}")

Total unique intents: 99
Train size: 4099, Validation size: 724


**2. Knowledge Base (Intent to Answer Mapping)**
We create a dictionary to map each intent to its standard answer.

In [11]:
# Mapping Intent to canonical Answer (assuming 1 answer per intent usually)
intent_to_answer = {}
for idx, row in df.iterrows():
    intent = row['intent']
    if intent not in intent_to_answer:
        intent_to_answer[intent] = row['answer']

print(f"Total mapped answers: {len(intent_to_answer)}")

Total mapped answers: 99


**3. Tokenization (IndoBERT)**

In [12]:
import torch
from transformers import AutoTokenizer

model_checkpoint = "indobenchmark/indobert-base-p1"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=128)

class IntentDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = IntentDataset(train_encodings, train_labels)
val_dataset = IntentDataset(val_encodings, val_labels)

**4. Model Training**

In [13]:
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments
import numpy as np
from sklearn.metrics import accuracy_score

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {'accuracy': accuracy_score(p.label_ids, preds)}

model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint, 
    num_labels=num_labels
)

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=50,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

RuntimeError: Failed to import transformers.trainer because of the following error (look up to see its traceback):
Failed to import transformers.integrations.peft because of the following error (look up to see its traceback):
module 'numpy' has no attribute 'dtypes'

**5. Evaluation Dashboard**

In [ ]:
eval_results = trainer.evaluate()
print(f"Validation Accuracy: {eval_results['eval_accuracy'] * 100:.2f}%")

**6. Simulate Chatbot Q&A Pipeline (CAG Approach)**

This pipeline will classify user's query intent using the model, grab the standardized answer mapping, and optionally formulate the response.

In [ ]:
import torch.nn.functional as F

def get_bot_response(query):
    # 1. Tokenize query
    inputs = tokenizer(query, return_tensors="pt", truncation=True, padding=True, max_length=128)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    # 2. Predict with Model
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = F.softmax(logits, dim=1)
        
    # 3. Get Predicted Intent
    predicted_class_id = torch.argmax(probs, dim=1).item()
    confidence_score = probs[0][predicted_class_id].item()
    
    predicted_intent = label_encoder.inverse_transform([predicted_class_id])[0]
    
    # 4. Confidence Threshold Check
    if confidence_score < 0.4:
        return "Maaf, saya kurang mengerti maksud Anda. Silakan coba gunakan bahasa yang lebih spesifik.", predicted_intent, confidence_score
    
    # 5. Retrieve Answer from Knowledge Base (Augmented Generation)
    bot_answer = intent_to_answer.get(predicted_intent, "Mohon maaf, jawaban untuk pertanyaan ini belum ada di sistem.")
    
    return bot_answer, predicted_intent, confidence_score

# -- Let's Test It! --
test_queries = [
    "Gimana cara bikin akun sih?",
    "Aku mau ganti password dong karena lupa",
    "Apa aja syarat dokumen KYC?",
    "Bisakah buat event banyak-banyak?"
]

print("========== Chatbot Q&A Simulation ==========\n")
for q in test_queries:
    print(f"User   : {q}")
    ans, intent, conf = get_bot_response(q)
    print(f"Bot    : {ans}")
    print(f"[Debug: Intent='{intent}', Confidence={conf:.4f}]\n")